# Running our Model
In this lesson we will take our neural network which we fine tuned in Google Colab and make sure it runs on our Raspberry Pi.

## Uploading Our Files
Make sure that you have downloaded the following files from your Google Drive:

- `model.pte`: A special file from executorch which has the neural network structure and parameters
- `classes.json`: A text file which has the names of our classes
- `eval_dataset.pkl`: A special file which has our evaluation dataset images

To upload these files:

1. In the file explorer double click on the `model-files` folder
2. Then click the upload button in the file explorer:  
   ![Upload button](./assets/running-our-model/jupyterlab-upload-button.png)
3. Select the file you want to upload

Complete the above steps once for each of the files.

# Setup
To set up this notebook run the code in the cell below. This will import some helpers we need to run our neural network:

_Run cell below_

In [ ]:
# RUN ME
import pickle
from IPython.display import display
import json
import pickle

import torch
from torch import nn
from torchvision import transforms
from executorch.runtime import Runtime
import numpy as np

# Loading our Model
First we are going to load the strucure of our neural network from our `model.pte` file.

We will use the `executorch` library to do this.

- First we'll create a `Runtime` class
- This `Runtime` class is able to load our `model.pte` file by calling the `load_program` function
- Once we have loaded the file we will retrieve the `forward` function from our model
  - If you remember back to when we were making our own neural network we defined a function named `forward` which runs the input data through the neurons
  - The MobileNet 2 neural network has the same thing
  - It defines a function named `forward` which runs the input image through the neural network

_Run the cell below_

In [ ]:
# RUN ME
runtime = Runtime.get()

program = runtime.load_program("./model-files/model.pte")
model = program.load_method("forward")

# Loading our Class Labels
Next we will load the names of our classes.

- Remember that our neural network outputs a list of the probability that the image contains each class
- The item in the list with the largest number is the class which is in the image
- Our classes list has the names of these classes

To load our `classes.json` text file we will call `json.load`:

_Run the cell below_

In [ ]:
# RUN ME
classes = []
with open("./model-files/classes.json", "r") as f:
    classes = json.load(f)

print("Loaded classes", classes)

# Setting up Our Dataset
Just like the fine tuning our model, we need to normalize our images before we can input them into our neural network.

These are the same transforms which we defined in our Google Colab notebook:

_Run the cell below_

In [ ]:
# RUN ME
mobilenet_mean = [0.485, 0.456, 0.406]
mobilenet_std = [0.229, 0.224, 0.225]

validation_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mobilenet_mean,
        std=mobilenet_std,
    )
])

unnormalize_transform = transforms.Normalize(
    mean=[-m / s for m, s in zip(mobilenet_mean, mobilenet_std)],
    std=[1 / s for s in mobilenet_std]
)

Now we will load our `eval_dataset.pkl`:

_Run the cell below_

In [ ]:
# RUN ME
with open("model-files/eval_dataset.pkl", "rb") as f:
    eval_dataset = pickle.load(f)

# Running Model on Evaluation Dataset 
Now let's run our neural network against the images in the evaluation dataset.

- We define the same `imshow` helper function that we used in Google Colab
- Then for each image in our evaluation dataset we ask the neural network to predict what class is in the photo

If all is working your neural network should correctly predict most of the classes in your evaluation dataset.

_Run the code below_

In [ ]:
# RUN ME
def imshow(inp, title = None):
    # Un-normalize image pixel data
    inp = unnormalize_transform(inp)

    # Flip pixel channels into correct value for displaying
    inp = inp.numpy().transpose((1, 2, 0))

    # Ensure values aren't too big or small to display
    inp = np.clip(inp, 0, 1)

    # Show image
    display(transforms.ToPILImage()(inp))

    # Show title
    if title is not None:
        display(title)

# Variables for tallying up the results
device = "cpu"
correct = 0
wrong = 0
num_shown = 0

# For each image
for img, class_ in eval_dataset:
    # We give the image to the neural network
    prediction = model.execute(img.unsqueeze(0).to(device))
    probabilities = nn.functional.softmax(prediction[0], dim=0).squeeze(0)

    # Get the predicted class
    predicted_class = probabilities.argmax().item()
    predicted_label = classes[predicted_class]
    prediction_probability = probabilities[predicted_class]

    actual_label = classes[class_]

    # Tally if we are correct or not
    if predicted_class != class_:
        wrong += 1
    else:
        correct += 1

    # Show image
    title = f"^^^ Predicted: {predicted_label} ({prediction_probability.item():0.2f}), Actual: {actual_label} ^^^"
    imshow(img, title)
    num_shown += 1

# Show totals
print(f"correct={correct}, wrong={wrong}")